# 01 – Data Import

In diesem Notebook wird der Rohdatensatz `Womens Clothing E-Commerce Reviews.csv` (Quelle: Kaggle, siehe Kapitel 3.2 der Arbeit) eingelesen. Anschliessend wird ein erster Überblick über die Struktur, die Datentypen und die fehlenden Werte des Datensatzes gegeben. Das Ergebnis wird als Rohversion unter `data/processed/` gespeichert und im Notebook `02_Data_Cleaning.ipynb` weiterverarbeitet.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Pfade

Hier werden die Verzeichnispfade relativ zum Projektroot definiert, damit das Notebook unabhängig vom lokalen Speicherort auf jedem Rechner lauffähig ist.

In [2]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DIR / "Womens Clothing E-Commerce Reviews.csv"
RAW_FILE

PosixPath('/Users/laraeibel/Desktop/Bachelorarbeit_Python/data/raw/Womens Clothing E-Commerce Reviews.csv')

## Einlesen

Der Rohdatensatz wird eingelesen und die Dimensionen des resultierenden DataFrames (Zeilen, Spalten) ausgegeben.

In [3]:
df = pd.read_csv(RAW_FILE)
df.shape

(23486, 11)

In [4]:
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Unnamed: 0               23486 non-null  int64
 1   Clothing ID              23486 non-null  int64
 2   Age                      23486 non-null  int64
 3   Title                    19676 non-null  str  
 4   Review Text              22641 non-null  str  
 5   Rating                   23486 non-null  int64
 6   Recommended IND          23486 non-null  int64
 7   Positive Feedback Count  23486 non-null  int64
 8   Division Name            23472 non-null  str  
 9   Department Name          23472 non-null  str  
 10  Class Name               23472 non-null  str  
dtypes: int64(6), str(5)
memory usage: 2.0 MB


## Erste Bereinigung: Index-Spalte

Beim Export aus dem Ursprungsformat wurde der Zeilenindex automatisch als Spalte `Unnamed: 0` übernommen. Diese Spalte wird hier in `Review ID` umbenannt, um sie als aussagekräftige Kennung nutzen zu können.

In [6]:
# "Unnamed: 0" ist der Zeilenindex aus dem Original-Export -> in aussagekräftige ID umbenennen
df = df.rename(columns={"Unnamed: 0": "Review ID"})
df.columns.tolist()

['Review ID',
 'Clothing ID',
 'Age',
 'Title',
 'Review Text',
 'Rating',
 'Recommended IND',
 'Positive Feedback Count',
 'Division Name',
 'Department Name',
 'Class Name']

## Fehlende Werte

Die Tabelle zeigt die Anzahl und den Anteil fehlender Werte je Spalte. Besonders betroffen sind `Title` (16,2 %) und `Review Text` (3,6 %). Alle übrigen Spalten weisen kaum oder keine fehlenden Werte auf.


In [7]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "missing_%": missing_pct})

,missing,missing_%
Title,3810,16.22
Review Text,845,3.60
Division Name,14,0.06
Department Name,14,0.06
Class Name,14,0.06
Review ID,0,0.00
Clothing ID,0,0.00
Age,0,0.00
Rating,0,0.00
Recommended IND,0,0.00


## Deskriptive Übersicht

Die deskriptive Statistik gibt einen ersten Überblick über die Variablen, darunter die Altersspanne von 18 bis 99 Jahren und die Verteilung der Sternebewertungen mit einem Median von 5.


In [8]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Review ID,23486.0,NaN,NaN,NaN,11742.5,6779.968547,0.0,5871.25,11742.5,17613.75,23485.0
Clothing ID,23486.0,NaN,NaN,NaN,918.118709,203.29898,0.0,861.0,936.0,1078.0,1205.0
Age,23486.0,NaN,NaN,NaN,43.198544,12.279544,18.0,34.0,41.0,52.0,99.0
Title,19676,13993,Love it!,136,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Review Text,22641,22634,Perfect fit and i've gotten so many compliment...,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rating,23486.0,NaN,NaN,NaN,4.196032,1.110031,1.0,4.0,5.0,5.0,5.0
Recommended IND,23486.0,NaN,NaN,NaN,0.822362,0.382216,0.0,1.0,1.0,1.0,1.0
Positive Feedback Count,23486.0,NaN,NaN,NaN,2.535936,5.702202,0.0,0.0,1.0,3.0,122.0
Division Name,23472,3,General,13850,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Department Name,23472,6,Tops,10468,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Speichern für die weitere Verarbeitung
Der vollständige, aber umbenannte Datensatz wird gespeichert und an `02_Data_Cleaning.ipynb` zur weiteren Bereinigung übergeben.

In [9]:
out_path = PROCESSED_DIR / "reviews_imported.csv"
df.to_csv(out_path, index=False)
out_path

PosixPath('/Users/laraeibel/Desktop/Bachelorarbeit_Python/data/processed/reviews_imported.csv')